# Orbital Debris SQL Queries

This notebook is intentionally SQL-first and contains query work only (no visualizations).

## **Setup And Function Declarations**

In [1]:
import pandas as pd
import sqlite3

def run_query(sql):
    return pd.read_sql(sql, conn)

pd.set_option('display.max_rows', 100)

conn = sqlite3.connect('../data/clean/orbital_debris.db')

## Primary Question 1: Growth and Decoupling Baseline

In [2]:
# I absolutely detest table aliases in SQL. I find them to be more confusing 
# than helpful, especially when the table names are not that long to begin with.
# I understand that they can be useful in some cases, but I prefer to just write 
# out the full table names for clarity. Nested queries are usually the only time
# I find them to be necessary, and even then I try to avoid them if possible.

q1 = '''
WITH yearly AS (
  SELECT
    launch_events.launch_year AS launch_year,
    COUNT(*) AS objects_launched,
    COUNT(DISTINCT launch_events.launch_id) AS launch_missions,
    SUM(
      CASE
        WHEN UPPER(COALESCE(satellites.object_type, '')) = 'PAYLOAD' THEN 1
        ELSE 0
      END
    ) AS payload_objects
  FROM satellites
  JOIN launch_events ON launch_events.launch_id = satellites.launch_id
  WHERE launch_events.launch_year IS NOT NULL
  GROUP BY launch_events.launch_year
)
SELECT
  launch_year,
  objects_launched,
  launch_missions,
  payload_objects,
  ROUND(100.0 * payload_objects / NULLIF(objects_launched, 0), 2) AS payload_share_pct,
  SUM(objects_launched) OVER (ORDER BY launch_year) AS cumulative_objects
FROM yearly
ORDER BY launch_year;
'''
launch_trend = run_query(q1)
launch_trend.head(100)

,launch_year,objects_launched,launch_missions,payload_objects,payload_share_pct,cumulative_objects
0,1958,3,1,1,33.33,3
1,1959,7,5,5,71.43,10
2,1960,13,6,5,38.46,23
3,1961,213,8,9,4.23,236
4,1962,33,15,14,42.42,269
5,1963,91,12,17,18.68,360
6,1964,60,21,28,46.67,420
7,1965,449,33,52,11.58,869
8,1966,207,34,38,18.36,1076
9,1967,95,31,49,51.58,1171


## **Primary Question 2: High-Risk Distribution by Altitude Band**

## **Primary Question 3: Zombie Concentration by Owner**

## **Secondary: Object Type × Operational Status**

## **Secondary: User Category Profile**

## **Extra Questions!**

## **Tidy Up!**

In [3]:
# Close the connection.
# Release resources and ensure clean exit (old habits die hard).
conn.close()
print('Connection closed.')

Connection closed.
